In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Load Dataset

In [ ]:
#=====================================
# Loading total training data
#=====================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

#train data
#----------------------------
df = pd.read_csv("/kaggle/input/competitions/histopathologic-cancer-detection/train_labels.csv")
train_dir = "/kaggle/input/competitions/histopathologic-cancer-detection/train/"
df["image_path"] = train_dir + df["id"] + ".tif"
df.head() 

In [ ]:
print(f'value count of labels :{df['label'].value_counts()}')
print(f'shape of dataset:{df.shape}') 

In [ ]:
row = df.iloc[0] 
image_id = row["id"]
label = row["label"] 

image_path = row['image_path']
image = Image.open(image_path).convert("RGB") 


print("Image ID:", image_path)
print("Label:", label)
print("Image shape:", image.size)  

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4)) 

for i in range(4):

    row = df.iloc[i]

    image_path = row["image_path"]
    label = row["label"]

    image = Image.open(image_path).convert("RGB")

    axes[i].imshow(image)
    axes[i].set_title(f"Label: {label}")
    axes[i].axis("off")
   

plt.show()

In [ ]:
#=====================================
# Loading total test data  
#=====================================
import pandas as pd


test_dir = "/kaggle/input/competitions/histopathologic-cancer-detection/test/"

test_df = pd.read_csv(
    "/kaggle/input/competitions/histopathologic-cancer-detection/sample_submission.csv"
)

test_df["image_path"] = test_dir + test_df["id"] + ".tif"

test_df.tail(10)  

# Transform and DataLoader

In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image 

class CancerDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        image_path = self.dataframe.iloc[index]["image_path"]
        label = self.dataframe.iloc[index]["label"]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(
            label,
            dtype=torch.float32
        )

        return image, label 
   

In [ ]:
#=====================================
# data split 
#=====================================
from sklearn.model_selection import train_test_split 

train_df, val_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(f'train dataset :{len(train_df)}')
print(f'val dataset :{len(val_df)}') 

In [ ]:
#=====================================
# transform train data 
#=====================================
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5), 
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

In [ ]:
#=====================================
# transform val data 
#=====================================
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
]) 

In [ ]:
train_dataset = CancerDataset(
    train_df,
    transform=train_transform
)

val_dataset = CancerDataset(
    val_df,
    transform=val_transform
) 

In [ ]:
#=====================================
# Train and Val Data loader  
#=====================================
from torch.utils.data import DataLoader 
train_loader=DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)
val_loader=DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False
)


In [ ]:
print(f'length of train Loader:{len(train_loader)}') 
print(f'length of val Loader:{len(val_loader)}') 


# Model Building 

In [ ]:
print(f'GPU is available:{torch.cuda.is_available()}')  

In [ ]:
#Image shape: (96, 96) 

# =============================
# Define CNN from Scratch
# =============================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device :", device) 


import torch.nn as nn
class NewCNN(nn.Module):
  def __init__(self):
    super(NewCNN,self).__init__()
    self.conv1=nn.Conv2d(
        in_channels=3,
        out_channels=16,
        kernel_size=3,
        stride=1,
        padding=1
    )
    self.relu1=nn.ReLU()
    self.pool1=nn.MaxPool2d(kernel_size=2,stride=2) #48
    self.conv2=nn.Conv2d(in_channels=16,
                         out_channels=32,
                         kernel_size=3,
                         stride=1,
                         padding=1
    )
    self.relu2=nn.ReLU()
    self.pool2=nn.MaxPool2d(kernel_size=2,stride=2) #24

    self.conv3=nn.Conv2d(in_channels=32,
                         out_channels=64,
                         kernel_size=3,
                         stride=1,
                         padding=1
    )
    self.relu3=nn.ReLU() 
    self.pool3=nn.MaxPool2d(kernel_size=2,stride=2) #12

    
      
    self.fc1=nn.Linear(in_features=64*12*12,out_features=128) 
    self.relu4=nn.ReLU() 
    
    self.dropout=nn.Dropout(p=0.4) 
    self.fc2=nn.Linear(in_features=128,out_features=64)  
    self.fc3=nn.Linear(in_features=64,out_features=1) 
  
  def forward(self,x):
    x=self.conv1(x)
    x=self.relu1(x)
    x=self.pool1(x) #48,48
    x=self.conv2(x)
    x=self.relu2(x)
    x=self.pool2(x) #24,24
    x=self.conv3(x)
    x=self.relu3(x)
    x=self.pool3(x)  #12,12 
     
    x=x.view(x.size(0),-1)
    x=self.fc1(x)
    x=self.relu4(x)
    x=self.dropout(x)
    x=self.fc2(x)
    x=self.fc3(x)
    return x
model=NewCNN().to(device)
print(model) 


In [ ]:
# =============================
# LOSS FN and Optimizer
# =============================
loss_fn = nn.BCEWithLogitsLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) 


In [ ]:
# =============================
# Train one Epoch FN
# =============================

def train_one_epoch(model,train_loader,loss_fn,optimizer):
  model.train()
  total_loss=0
  correct=0
  total=0
  for image,label in train_loader:
    image=image.to(device)
    label=label.float().to(device)
    optimizer.zero_grad()
   
    output = model(image).squeeze(1) # [150,1] → [150]
    loss=loss_fn(output,label)
    loss.backward() 
    optimizer.step()
    total_loss+=loss.item()  
    probability = torch.sigmoid(output)
    predicted = (probability >= 0.5).float()
    
    correct+=(predicted==label).sum().item() 
    total+=len(label)

  avg_loss=total_loss/len(train_loader)
  accuracy=(correct/total)*100
  return avg_loss,accuracy

In [ ]:
# =============================
# Validation FN
# =============================

def validation(model,val_loader,loss_fn):
  model.eval()
  total_loss=0
  correct=0
  total=0
  with torch.no_grad():
    for image,label in val_loader:
      image=image.to(device)
      label=label.float().to(device) 
    
      output = model(image).squeeze(1) # [150,1] → [150]
      loss=loss_fn(output,label)
      total_loss+=loss.item() 

      probability = torch.sigmoid(output)
      predicted = (probability >= 0.5).float()
        
     
      correct+=(predicted==label).sum().item() 
      total+=len(label)
  avg_loss=total_loss/len(val_loader)  
  accuracy=(correct/total)*100
  return avg_loss,accuracy

# CNN Model Training 

In [ ]:
# =============================
# Training Model for 15 epoch
# =============================
num_epoch=15 
train_loss=[]
val_loss=[]
train_acc=[]
val_acc=[]
for epoch in range(num_epoch):
  train_epoch_loss,train_epoch_acc=train_one_epoch(model,train_loader,loss_fn,optimizer)
  val_epoch_loss,val_epoch_acc=validation(model,val_loader,loss_fn)
  train_loss.append(train_epoch_loss)
  val_loss.append(val_epoch_loss)
  train_acc.append(train_epoch_acc)
  val_acc.append(val_epoch_acc)
  print(f"train loss is : {train_epoch_loss} and train accuracy is : {train_epoch_acc}")
  print(f"val loss is : {val_epoch_loss} and val accuracy is : {val_epoch_acc}")

print("training done ") 

In [ ]:
# =============================
# val roc-auc score 
# =============================
from sklearn.metrics import roc_auc_score

model.eval()

actual_labels = []
predicted_probabilities = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)

        #  model output
        outputs = model(images).squeeze(1)

       
        probabilities = torch.sigmoid(outputs)

        actual_labels.extend(
            labels.cpu().numpy()
        )

        predicted_probabilities.extend(
            probabilities.cpu().numpy()
        )

# validation ROC-AUC
val_auc = roc_auc_score(
    actual_labels,
    predicted_probabilities
)

print("Validation ROC-AUC of model:", val_auc)

# EfficientNet_B0

In [ ]:
# =============================
# Loading EfficientNet_B0
# =============================

from torchvision import models 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weights=models.EfficientNet_B0_Weights.DEFAULT
model_2=models.efficientnet_b0(weights=weights).to(device)  

In [ ]:
print(f"Model structure (abbreviated):")
print(f"  model.features  → {len(model_2.features)} blocks (the CNN feature extractor)")
print(f"  model.avgpool   → adaptive average pooling")
print(f"  model.classifier → {model_2.classifier}")

total_params = sum(p.numel() for p in model_2.parameters())
print(f"\nTotal parameters: {total_params:,}")
print(f"Currently outputs: {model_2.classifier[1].out_features} classes (ImageNet)") 

In [ ]:
in_features = model_2.classifier[1].in_features
# Binary-classification output
model_2.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(
        in_features=in_features,
        out_features=1
    )
)

# every layer is trainable
for param in model_2.parameters():
    param.requires_grad = True 

model_2=model_2.to(device)

In [ ]:
total = sum( 
    p.numel() for p in model_2.parameters()
)

trainable = sum(
    p.numel()
    for p in model_2.parameters()
    if p.requires_grad
)

frozen = total - trainable

print(f"Total parameters:     {total:,}")
print(f"Frozen parameters:    {frozen:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Training: {100 * trainable / total:.2f}%")
print(f"\nClassifier:\n{model_2.classifier}")

In [ ]:
#==========================================
# transform train data  for EfficientNet_B0
#==========================================

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5), 
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
]) 
train_dataset = CancerDataset(
    train_df,
    transform=train_transform
)

val_dataset = CancerDataset(
    val_df,
    transform=val_transform
) 
  

In [ ]:
# =============================
#  Data Loader for EfficientNet_B0
# =============================
from torch.utils.data import DataLoader 
train_loader=DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)
val_loader=DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
loss_fn = nn.BCEWithLogitsLoss() 
optimizer = torch.optim.AdamW(
    model_2.parameters(),
    lr=0.0001
) 

In [ ]:
# =============================
#  Full Fine-Tuning of Model
# =============================

num_epoch=6    

train_loss=[]
val_loss=[]
train_acc=[]
val_acc=[]
for epoch in range(num_epoch):
  train_epoch_loss,train_epoch_acc=train_one_epoch(model_2,train_loader,loss_fn,optimizer)
  val_epoch_loss,val_epoch_acc=validation(model_2,val_loader,loss_fn)
  train_loss.append(train_epoch_loss)
  val_loss.append(val_epoch_loss)
  train_acc.append(train_epoch_acc)
  val_acc.append(val_epoch_acc)
  print(f"train loss is : {train_epoch_loss} and train accuracy is : {train_epoch_acc}")
  print(f"val loss is : {val_epoch_loss} and val accuracy is : {val_epoch_acc}")

print("model_2 training done ") 

In [ ]:
# =============================
# val roc-auc score 
# =============================
from sklearn.metrics import roc_auc_score

model_2.eval()

actual_labels = []
predicted_probabilities = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)

        #  model output
        outputs = model_2(images).squeeze(1)

       
        probabilities = torch.sigmoid(outputs)

        actual_labels.extend(
            labels.cpu().numpy()
        )

        predicted_probabilities.extend(
            probabilities.cpu().numpy()
        )

# validation ROC-AUC
val_auc = roc_auc_score(
    actual_labels,
    predicted_probabilities
)

print("Validation ROC-AUC of model_2:", val_auc)


# Test Prediction & Submission

In [ ]:
#=====================================
# transform test data 
#=====================================

transform = transforms.Compose([
    transforms.ToTensor(),
     transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ) 
   
]) 

class Cancer_test_Dataset(Dataset):

    def __init__(self, dataframe):
        self.dataframe = dataframe

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        image_path = self.dataframe.iloc[index]["image_path"]
        image = Image.open(image_path).convert("RGB")
        image = transform(image)

        return image
        
test_data=Cancer_test_Dataset(test_df) 

In [ ]:
#=====================================
# test data loader
#=====================================

test_loader=DataLoader(
    test_data,
    batch_size=32,
    shuffle=False
) 
print(f'lenght of test loader: {len(test_loader)}')

In [ ]:
#=====================================
# test Prediction and Submission
#=====================================
model_2.eval()

predictions = []

with torch.no_grad():

    for images in test_loader:

        images = images.to(device)

        outputs = model_2(images).squeeze(1)
        probabilities = torch.sigmoid(outputs)

        predictions.extend(
            probabilities.cpu().numpy()
        )

submission = pd.DataFrame({
    "id": test_df["id"].values,
    "label": predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()

